# (IIP314W) Optimización Aplicada a Negocios
## Ayudantía 8: Problema Dual, Simplex Dual, Holguras Complementarias y Análisis de Sensibilidad

---

**Profesor:** Ing. Rodrigo Trigo Vilches  
**Ayudante:** Lic. Vicente Ramírez Almonacid  
**Fecha:** 29 de Abril, 2026  
**Universidad del Desarrollo**

---

## Sección 0 — Resumen Teórico

Esta ayudantía consolida cuatro conceptos que se desprenden naturalmente del Método Simplex: el **Problema Dual**, el **Algoritmo Simplex Dual**, las **Condiciones de Holgura Complementaria** y el **Análisis de Sensibilidad**. Todos ellos son herramientas que permiten extraer información adicional de la solución óptima sin resolver el problema desde cero.

### Tema 1: El Problema Dual

#### 1.1 Par Primal–Dual en Forma Canónica

A todo problema de programación lineal (**primal**) le corresponde un problema **dual** asociado. En su forma canónica de minimización:

| | Primal | Dual |
|:---:|:---|:---|
| **Objetivo** | $\min \quad c^\top x$ | $\max \quad b^\top y$ |
| **Restricciones** | $Ax \geq b$ | $A^\top y \leq c$ |
| **No negatividad** | $x \geq 0$ | $y \geq 0$ |

#### 1.2 Reglas de Construcción del Dual

Las siguientes reglas permiten construir el dual directamente desde el primal, sin necesidad de memorizar la forma canónica:

| Primal ($\min$) | Dual ($\max$) |
|:---|:---|
| Restricción $i$: $\geq b_i$ | Variable $y_i \geq 0$ |
| Restricción $i$: $\leq b_i$ | Variable $y_i \leq 0$ |
| Restricción $i$: $= b_i$ | Variable $y_i$ libre |
| Variable $x_j \geq 0$ | Restricción $j$: $\leq c_j$ |
| Variable $x_j \leq 0$ | Restricción $j$: $\geq c_j$ |
| Variable $x_j$ libre | Restricción $j$: $= c_j$ |

> **Nota práctica:** el primal de maximización con restricciones $\leq$ se transforma directamente en el dual de minimización con restricciones $\geq$, y viceversa. Siempre verificar que el número de variables del dual igual al número de restricciones del primal.

#### 1.3 Teorema de Dualidad Débil

Para cualquier $x$ factible en el primal y cualquier $y$ factible en el dual:

$$b^\top y \leq c^\top x$$

Esto implica que el valor objetivo del dual es siempre una **cota inferior** del valor objetivo del primal (en el caso min/max). En particular, si se encuentra un par $(x, y)$ factible tal que $b^\top y = c^\top x$, ambos son óptimos.

#### 1.4 Teorema de Dualidad Fuerte

Si el primal tiene solución óptima $x^*$, entonces el dual también tiene solución óptima $y^*$, y se cumple:

$$c^\top x^* = b^\top y^*$$

Es decir, en el óptimo los valores de ambos objetivos **coinciden exactamente**.

#### 1.5 Interpretación Económica: Precios Sombra

Las variables duales $y_i^*$ se denominan **precios sombra** (o precios duales) de las restricciones. Cada $y_i^*$ representa el **valor marginal** de relajar la restricción $i$ en una unidad:

$$y_i^* = \frac{\partial z^*}{\partial b_i}$$

Si la restricción $i$ está **inactiva** (holgura positiva), su precio sombra es $y_i^* = 0$: relajar esa restricción no mejora el óptimo. Si está **activa** (holgura cero), $y_i^* > 0$: incrementar $b_i$ en una unidad mejora $z^*$ en exactamente $y_i^*$ unidades.

#### Tabla de Relaciones de Dualidad en Programación Lineal

La siguiente tabla resume **todas** las correspondencias primal–dual, cubriendo los dos casos canónicos (primal de maximización y primal de minimización):

| | **Primal MAX** | **Dual MIN** |
|:---:|:---:|:---:|
| **Objetivo** | $\max\ c^\top x$ | $\min\ b^\top y$ |
| Restricción $i$: $\leq b_i$ | $\Leftrightarrow$ | Variable $y_i \geq 0$ |
| Restricción $i$: $\geq b_i$ | $\Leftrightarrow$ | Variable $y_i \leq 0$ |
| Restricción $i$: $= b_i$ | $\Leftrightarrow$ | Variable $y_i$ libre |
| Variable $x_j \geq 0$ | $\Leftrightarrow$ | Restricción $j$: $\geq c_j$ |
| Variable $x_j \leq 0$ | $\Leftrightarrow$ | Restricción $j$: $\leq c_j$ |
| Variable $x_j$ libre | $\Leftrightarrow$ | Restricción $j$: $= c_j$ |

| | **Primal MIN** | **Dual MAX** |
|:---:|:---:|:---:|
| **Objetivo** | $\min\ c^\top x$ | $\max\ b^\top y$ |
| Restricción $i$: $\geq b_i$ | $\Leftrightarrow$ | Variable $y_i \geq 0$ |
| Restricción $i$: $\leq b_i$ | $\Leftrightarrow$ | Variable $y_i \leq 0$ |
| Restricción $i$: $= b_i$ | $\Leftrightarrow$ | Variable $y_i$ libre |
| Variable $x_j \geq 0$ | $\Leftrightarrow$ | Restricción $j$: $\leq c_j$ |
| Variable $x_j \leq 0$ | $\Leftrightarrow$ | Restricción $j$: $\geq c_j$ |
| Variable $x_j$ libre | $\Leftrightarrow$ | Restricción $j$: $= c_j$ |

> **Regla nemotécnica:** al pasar de MAX a MIN (o viceversa), el sentido de las desigualdades de restricciones y el signo de las variables se **invierten**. La relación es simétrica: el dual del dual es el primal.

### Tema 2: Algoritmo Simplex Dual

#### 2.1 Diferencia Clave Respecto al Simplex Primal

El **Simplex Primal** parte de una solución *factible* (RHS $\geq 0$) y busca la *optimalidad* (costos reducidos $\geq 0$).  
El **Simplex Dual** parte de una solución *dual-factible* (costos reducidos $\geq 0$ ya satisfechos) pero *primal-infactible* (algún RHS $< 0$), y trabaja para recuperar la factibilidad.

Esto ocurre naturalmente cuando un problema de minimización con restricciones $\geq$ se plantea directamente en tableau (las holguras son negativas).

#### 2.2 Reglas de Pivoteo Dual

| Paso | Criterio |
|:---:|:---|
| **Variable que sale** | La variable básica con el RHS más negativo (fila más infactible). |
| **Variable que entra** | La variable no básica que minimiza el cociente $\left|\dfrac{\bar{c}_j}{a_{rj}}\right|$ para las columnas con $a_{rj} < 0$ en la fila pivote $r$. |
| **Optimalidad** | Se alcanza cuando todos los RHS son $\geq 0$ (la solución se vuelve factible). |
| **Infactibilidad** | Si en alguna iteración una fila pivote tiene todos sus coeficientes $\geq 0$ pero RHS $< 0$, el problema primal es **infactible**. |

#### 2.3 Tableau del Simplex Dual

El tableau es el mismo que el del Simplex Primal. La diferencia está en **qué elemento se elige como pivote**:

$$\text{Entra: } \arg\min_{j:\, a_{rj}<0} \left|\frac{\bar{c}_j}{a_{rj}}\right| \qquad \text{Sale: } r = \arg\min_i \{\bar{b}_i \mid \bar{b}_i < 0\}$$

> La condición de entrada busca preservar la **dual-factibilidad**: que los costos reducidos permanezcan $\geq 0$ tras el pivoteo.

### Tema 3: Condiciones de Holgura Complementaria

#### 3.1 Enunciado

Para cualquier par de soluciones óptimas $x^*$ (primal) e $y^*$ (dual), se cumplen las siguientes condiciones para **todo** $i$ y $j$:

$$y_i^* \cdot \underbrace{\left(a_i^\top x^* - b_i\right)}_{\text{holgura primal}} = 0 \qquad \forall i$$

$$x_j^* \cdot \underbrace{\left(c_j - a_j^\top y^*\right)}_{\text{holgura dual}} = 0 \qquad \forall j$$

En palabras:
- Si la restricción primal $i$ **no está activa** ($a_i^\top x^* > b_i$), entonces $y_i^* = 0$.
- Si $y_i^* > 0$, entonces la restricción primal $i$ **debe estar activa** ($a_i^\top x^* = b_i$).
- Si $x_j^* > 0$, entonces la restricción dual $j$ **debe estar activa** ($a_j^\top y^* = c_j$).
- Si la restricción dual $j$ **no está activa** ($a_j^\top y^* < c_j$), entonces $x_j^* = 0$.

#### 3.2 Uso Práctico: Recuperar la Solución Primal desde el Dual

Si se conoce la solución óptima del dual $y^*$:

1. Identificar qué $y_i^* > 0$: las restricciones primales correspondientes son activas ($=b_i$).
2. Identificar qué $y_i^* = 0$: las restricciones correspondientes pueden tener holgura.
3. Usar las restricciones activas como sistema de ecuaciones para encontrar $x^*$.
4. Verificar que $x^* \geq 0$ y que las restricciones inactivas no sean violadas.

### Tema 4: Análisis de Sensibilidad

El análisis de sensibilidad responde a la pregunta: **¿en qué rango puede variar un parámetro del problema sin que la base óptima actual deje de ser óptima?**

Sea $B$ la base óptima con $B^{-1}$ conocida.

#### 4.1 Sensibilidad sobre $b$ (lado derecho / RHS)

Si se perturba el RHS como $b \to b + \Delta e_i$ (modificar $b_i$ en $\Delta$), la solución básica actualizada es:

$$x_B = B^{-1}(b + \Delta e_i) = B^{-1}b + \Delta B^{-1}e_i = \bar{b} + \Delta d_i$$

donde $d_i$ es la $i$-ésima columna de $B^{-1}$. La base sigue siendo factible mientras:

$$\bar{b} + \Delta d_i \geq 0 \qquad \Rightarrow \qquad \Delta \geq -\frac{\bar{b}_k}{(d_i)_k} \text{ para } (d_i)_k > 0 \quad \text{ y } \quad \Delta \leq -\frac{\bar{b}_k}{(d_i)_k} \text{ para } (d_i)_k < 0$$

El **precio sombra** de la restricción $i$ es $y_i^* = c_B^\top B^{-1} e_i$, y representa la tasa de cambio de $z^*$ con respecto a $b_i$.

#### 4.2 Sensibilidad sobre $c$ (coeficientes del objetivo)

Si se perturba el coeficiente objetivo de la variable **no básica** $x_j$ como $c_j \to c_j + \Delta$, el costo reducido de $x_j$ cambia:

$$\bar{c}_j + \Delta \geq 0 \qquad \Rightarrow \qquad \Delta \geq -\bar{c}_j$$

Si se perturba el coeficiente de la variable **básica** $x_k$ (que está en la posición $p$ de la base), el vector $c_B$ cambia y todos los costos reducidos de las no básicas se ven afectados:

$$\bar{c}_j^{\text{nuevo}} = \bar{c}_j - \Delta (B^{-1}A_{NB})_{p,j} \geq 0 \qquad \forall j \notin B$$

Esto define un sistema de inecuaciones en $\Delta$ cuya intersección es el rango de optimalidad.

#### 4.3 Interpretación del Precio Sombra

El precio sombra $y_i^*$ de la restricción $i$ tiene unidades de **[unidades de objetivo] / [unidades de $b_i$]**. Por ejemplo, si $b_i$ es horas disponibles y $z$ es beneficio en pesos, entonces $y_i^*$ es el beneficio adicional por hora extra. Este valor es válido **solo dentro del rango de sensibilidad** de $b_i$.

---

### Ejemplo Ilustrativo: Construcción del Par Primal–Dual

Consideremos el siguiente problema sencillo para ilustrar cómo se construye el dual paso a paso.

**Problema Primal:**

$$\max \quad z = 2x_1 + 3x_2$$

$$\text{s.a.} \quad \begin{cases} x_1 + x_2 \leq 4 \\ x_1 + 2x_2 \leq 6 \\ x_1,\, x_2 \geq 0 \end{cases}$$

**Paso 1 — Identificar el tipo de problema y restricciones:**

- Objetivo: maximización → el dual será de **minimización**
- 2 restricciones $\leq$ → 2 variables duales $y_1, y_2 \geq 0$
- 2 variables $x_j \geq 0$ → 2 restricciones duales de tipo $\geq$

**Paso 2 — Construir la función objetivo dual** (coeficientes del RHS del primal):

$$\min \quad w = 4y_1 + 6y_2$$

**Paso 3 — Construir las restricciones duales** (una por cada variable primal; los coeficientes se leen por **columnas** de la matriz $A$):

| Variable primal | Columna en $A$ | Restricción dual | RHS |
|:---:|:---:|:---:|:---:|
| $x_1$ | $(1,\; 1)$ | $y_1 + y_2 \geq$ | $2$ |
| $x_2$ | $(1,\; 2)$ | $y_1 + 2y_2 \geq$ | $3$ |

**Problema Dual resultante:**

$$\min \quad w = 4y_1 + 6y_2$$

$$\text{s.a.} \quad \begin{cases} y_1 + y_2 \geq 2 \\ y_1 + 2y_2 \geq 3 \\ y_1,\, y_2 \geq 0 \end{cases}$$

**Paso 4 — Verificar la dualidad fuerte** (resolución manual):

*Primal:* en el óptimo, ambas restricciones están activas (vértice del poliedro):
$$x_1 + x_2 = 4 \quad \text{y} \quad x_1 + 2x_2 = 6 \quad \Rightarrow \quad x_1^* = 2,\; x_2^* = 2, \quad z^* = 2(2)+3(2) = \mathbf{10}$$

*Dual:* por holguras complementarias, como $x_1^*, x_2^* > 0$, ambas restricciones duales son activas:
$$y_1 + y_2 = 2 \quad \text{y} \quad y_1 + 2y_2 = 3 \quad \Rightarrow \quad y_1^* = 1,\; y_2^* = 1, \quad w^* = 4(1)+6(1) = \mathbf{10}$$

$$\boxed{z^* = w^* = 10 \quad \checkmark \text{ Dualidad Fuerte}}$$

> **Lectura económica:** $y_1^* = 1$ indica que una unidad extra de la primera restricción (capacidad 4) aumenta el valor óptimo en $1$. Análogamente para $y_2^* = 1$ con la segunda restricción (capacidad 6).

---

---

## Ejercicio 1 — Planteamiento del Dual y Resolución

### Contexto de Negocio

**TechPrint S.A.** es una empresa de impresión digital que produce dos tipos de materiales publicitarios: **Afiches** ($x_1$, en cientos de unidades) y **Catálogos** ($x_2$, en cientos de unidades). La empresa opera con tres recursos limitados semanalmente:

| Recurso | Afiches ($x_1$) | Catálogos ($x_2$) | Disponible |
|:---|:---:|:---:|:---:|
| Tinta especial (litros) | $2$ | $3$ | $\leq 18$ L |
| Tiempo de prensa (horas) | $4$ | $2$ | $\leq 20$ hrs |
| Papel recubierto (resmas) | $1$ | $3$ | $\leq 15$ resmas |
| **Margen neto (\$/cien unid.)** | **\$5** | **\$4** | — |

El gerente de producción desea maximizar el margen neto semanal.

### Formulación Primal

$$\max \quad z = 5x_1 + 4x_2$$

$$\text{s.a.} \quad \begin{cases}
2x_1 + 3x_2 \leq 18 & \text{(tinta)} \\
4x_1 + 2x_2 \leq 20 & \text{(prensa)} \\
x_1 + 3x_2 \leq 15 & \text{(papel)} \\
x_1,\, x_2 \geq 0
\end{cases}$$

### Parte (a): Plantee el Problema Dual

El primal es de maximización con restricciones $\leq$ y variables $\geq 0$. Usando la tabla de dualidad:

- Cada restricción $\leq$ del primal genera una variable dual $y_i \geq 0$.
- Cada variable $x_j \geq 0$ del primal genera una restricción $\geq c_j$ en el dual.
- El dual de un problema de maximización es de minimización.

El primal tiene **3 restricciones** $\Rightarrow$ el dual tiene **3 variables** $(y_1, y_2, y_3)$.

El primal tiene **2 variables** $\Rightarrow$ el dual tiene **2 restricciones**.

$$\min \quad w = 18y_1 + 20y_2 + 15y_3$$

$$\text{s.a.} \quad \begin{cases}
2y_1 + 4y_2 + y_3 \geq 5 & \text{(columna de } x_1\text{)} \\
3y_1 + 2y_2 + 3y_3 \geq 4 & \text{(columna de } x_2\text{)} \\
y_1,\, y_2,\, y_3 \geq 0
\end{cases}$$

> **Interpretación:** $y_i$ es el precio sombra del recurso $i$. El dual busca el mínimo "costo imputado" de los recursos tal que cada unidad de producto esté valorada al menos en su margen neto.

### Parte (b): Resuelva el Dual por Simplex Tableau

El dual es un problema de **minimización con restricciones** $\geq$. Esto es exactamente el caso donde el **Simplex Dual** es natural: tras multiplicar cada restricción por $-1$ (para obtener $\leq$), las holguras serán negativas, dando una solución inicial dual-factible (costos reducidos $\geq 0$ para minimización) pero primal-infactible.

**Forma Estándar del Dual** (con variables de excedente $s_1, s_2 \geq 0$, multiplicando por $-1$ para usar holguras negativas como en el Simplex Dual):

$$\min \quad w = 18y_1 + 20y_2 + 15y_3 + 0s_1 + 0s_2$$

$$\text{s.a.} \quad \begin{cases}
2y_1 + 4y_2 + y_3 - s_1 = 5 \\
3y_1 + 2y_2 + 3y_3 - s_2 = 4 \\
y_1, y_2, y_3, s_1, s_2 \geq 0
\end{cases}$$

Para el Simplex Dual, multiplicamos las dos filas de restricciones por $-1$:

$$\begin{cases}
-2y_1 - 4y_2 - y_3 + s_1 = -5 \\
-3y_1 - 2y_2 - 3y_3 + s_2 = -4
\end{cases}$$

**Tableau Inicial (Iteración 0)**

Base inicial: $\{s_1, s_2\}$. Para minimización, la fila $w$ muestra los coeficientes de la función objetivo directamente (no negados). La condición de optimalidad dual es: todos los coeficientes en la fila $w$ para las no básicas sean $\geq 0$. ✓ (todos son $18, 20, 15 \geq 0$).

| Base | $y_1$ | $y_2$ | $y_3$ | $s_1$ | $s_2$ | RHS |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **$w$** | $18$ | $20$ | $15$ | $0$ | $0$ | $0$ |
| $s_1$ | $-2$ | $-4$ | $-1$ | $1$ | $0$ | $-5$ |
| $s_2$ | $-3$ | $-2$ | $-3$ | $0$ | $1$ | $-4$ |

**Criterio de salida (Simplex Dual):** el RHS más negativo es $-5$ (fila $s_1$) $\Rightarrow$ **sale $s_1$**.

**Criterio de entrada:** entre las columnas con coeficiente **negativo** en la fila de $s_1$, calcular:

$$\min_{j:\, a_{1j}<0} \left|\frac{w_j}{a_{1j}}\right| = \min\left(\left|\frac{18}{-2}\right|,\; \left|\frac{20}{-4}\right|,\; \left|\frac{15}{-1}\right|\right) = \min(9,\; 5,\; 15)$$

El mínimo es $5$ $\Rightarrow$ **entra $y_2$**. Elemento pivote: $\boxed{a_{1,y_2} = -4}$.

#### Iteración 1

**Operaciones de fila** (pivote en $a_{s_1, y_2} = -4$, fila de $s_1$):

$$R_{y_2}^{\text{nuevo}} = \frac{R_{s_1}}{-4} = \left[\frac{1}{2},\; 1,\; \frac{1}{4},\; -\frac{1}{4},\; 0 \;\middle|\; \frac{5}{4}\right]$$

$$R_w^{\text{nuevo}} = R_w - 20 \cdot R_{y_2}^{\text{nuevo}} = [18,20,15,0,0,0] - 20\cdot[\tfrac{1}{2},1,\tfrac{1}{4},-\tfrac{1}{4},0,\tfrac{5}{4}]$$
$$= [18-10,\;20-20,\;15-5,\;0+5,\;0,\;0-25] = [8,\;0,\;10,\;5,\;0,\;-25]$$

$$R_{s_2}^{\text{nuevo}} = R_{s_2} - (-2) \cdot R_{y_2}^{\text{nuevo}} = [-3,-2,-3,0,1,-4] + 2\cdot[\tfrac{1}{2},1,\tfrac{1}{4},-\tfrac{1}{4},0,\tfrac{5}{4}]$$
$$= [-3+1,\;-2+2,\;-3+\tfrac{1}{2},\;0-\tfrac{1}{2},\;1,\;-4+\tfrac{5}{2}] = [-2,\;0,\;-\tfrac{5}{2},\;-\tfrac{1}{2},\;1,\;-\tfrac{3}{2}]$$

| Base | $y_1$ | $y_2$ | $y_3$ | $s_1$ | $s_2$ | RHS |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **$w$** | $8$ | $0$ | $10$ | $5$ | $0$ | $\mathbf{-25}$ |
| $y_2$ | $1/2$ | $1$ | $1/4$ | $-1/4$ | $0$ | $5/4$ |
| $s_2$ | $-2$ | $0$ | $-5/2$ | $-1/2$ | $1$ | $-3/2$ |

El RHS de $s_2$ es $-3/2 < 0$ $\Rightarrow$ **sale $s_2$**.

**Criterio de entrada:** columnas con coeficiente negativo en fila $s_2$: $y_1$ y $y_3$ y $s_1$:

$$\min\left(\left|\frac{8}{-2}\right|,\; \left|\frac{10}{-5/2}\right|,\; \left|\frac{5}{-1/2}\right|\right) = \min(4,\; 4,\; 10)$$

Hay empate entre $y_1$ ($= 4$) e $y_3$ ($= 4$). Se elige $y_1$ (primera en aparecer). Elemento pivote: $\boxed{a_{s_2, y_1} = -2}$.

#### Iteración 2 — Solución Óptima

**Operaciones de fila** (pivote en $a_{s_2, y_1} = -2$, fila de $s_2$):

$$R_{y_1}^{\text{nuevo}} = \frac{R_{s_2}}{-2} = \left[1,\; 0,\; \frac{5}{4},\; \frac{1}{4},\; -\frac{1}{2} \;\middle|\; \frac{3}{4}\right]$$

$$R_w^{\text{nuevo}} = R_w - 8 \cdot R_{y_1}^{\text{nuevo}} = [8,0,10,5,0,-25] - 8\cdot[1,0,\tfrac{5}{4},\tfrac{1}{4},-\tfrac{1}{2},\tfrac{3}{4}]$$
$$= [8-8,\;0,\;10-10,\;5-2,\;0+4,\;-25-6] = [0,\;0,\;0,\;3,\;4,\;\mathbf{-31}]$$

$$R_{y_2}^{\text{nuevo}} = R_{y_2} - \frac{1}{2} \cdot R_{y_1}^{\text{nuevo}} = [\tfrac{1}{2},1,\tfrac{1}{4},-\tfrac{1}{4},0,\tfrac{5}{4}] - \tfrac{1}{2}\cdot[1,0,\tfrac{5}{4},\tfrac{1}{4},-\tfrac{1}{2},\tfrac{3}{4}]$$
$$= [\tfrac{1}{2}-\tfrac{1}{2},\;1,\;\tfrac{1}{4}-\tfrac{5}{8},\;-\tfrac{1}{4}-\tfrac{1}{8},\;0+\tfrac{1}{4},\;\tfrac{5}{4}-\tfrac{3}{8}]$$
$$= [0,\;1,\;-\tfrac{3}{8},\;-\tfrac{3}{8},\;\tfrac{1}{4},\;\tfrac{7}{8}]$$

| Base | $y_1$ | $y_2$ | $y_3$ | $s_1$ | $s_2$ | RHS |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **$w$** | $0$ | $0$ | $0$ | $3$ | $4$ | $\mathbf{-31}$ |
| $y_2$ | $0$ | $1$ | $-3/8$ | $-3/8$ | $1/4$ | $7/8$ |
| $y_1$ | $1$ | $0$ | $5/4$ | $1/4$ | $-1/2$ | $3/4$ |

**Todos los RHS son** $\geq 0$ ($7/8 > 0$, $3/4 > 0$): la solución es primal-factible. **Todos los coeficientes en la fila $w$ son** $\geq 0$: la solución es dual-óptima. $\Rightarrow$ **Solución óptima alcanzada.**

$$\boxed{y_1^* = \frac{3}{4}, \quad y_2^* = \frac{7}{8}, \quad y_3^* = 0, \quad w^* = 31}$$

> **Nota sobre el signo de $w$:** en el tableau de minimización con Simplex Dual, la fila $w$ acumula el valor de la función objetivo con signo negativo (análogamente al tableau de maximización). El valor real de $w^*$ es $|-31| = 31$. Esto se puede verificar: $w^* = 18 \cdot (3/4) + 20 \cdot (7/8) + 15 \cdot 0 = 13.5 + 17.5 + 0 = 31$. ✓

### Parte (c): Recupere la Solución Primal usando Holguras Complementarias

Conocemos la solución óptima dual: $y_1^* = 3/4$, $y_2^* = 7/8$, $y_3^* = 0$.

**Paso 1 — Identificar variables duales nulas:**

$y_3^* = 0 \Rightarrow$ la tercera restricción primal ($x_1 + 3x_2 \leq 15$) puede tener holgura (no necesariamente activa).

$y_1^* = 3/4 > 0$ y $y_2^* = 7/8 > 0$ $\Rightarrow$ las restricciones primales 1 y 2 **deben estar activas**:

$$2x_1 + 3x_2 = 18 \quad (1)$$
$$4x_1 + 2x_2 = 20 \quad (2)$$

**Paso 2 — Resolver el sistema de ecuaciones:**

De $(1)$: $2x_1 + 3x_2 = 18$

De $(2)$: $4x_1 + 2x_2 = 20 \Rightarrow 2x_1 + x_2 = 10 \Rightarrow x_2 = 10 - 2x_1$

Sustituyendo en $(1)$:
$$2x_1 + 3(10 - 2x_1) = 18 \Rightarrow 2x_1 + 30 - 6x_1 = 18 \Rightarrow -4x_1 = -12 \Rightarrow x_1 = 3$$

$$x_2 = 10 - 2(3) = 4$$

**Paso 3 — Verificar no negatividad y restricciones restantes:**

$x_1^* = 3 \geq 0$ ✓, $x_2^* = 4 \geq 0$ ✓

Restricción 3: $x_1 + 3x_2 = 3 + 12 = 15 \leq 15$ ✓ (coincidencia: también está activa, lo que es consistente con $y_3^* = 0$, pues la holgura complementaria solo exige que si $y_3^* = 0$ la restricción **puede** tener holgura, pero no que necesariamente la tenga).

$$\boxed{x_1^* = 3, \quad x_2^* = 4, \quad z^* = 5(3) + 4(4) = 15 + 16 = 31}$$

### Parte (d): Verificar la Dualidad Fuerte

El Teorema de Dualidad Fuerte garantiza que $z^* = w^*$ si ambos problemas son factibles y acotados.

$$z^* = 5x_1^* + 4x_2^* = 5(3) + 4(4) = 15 + 16 = \mathbf{31}$$

$$w^* = 18y_1^* + 20y_2^* + 15y_3^* = 18\left(\frac{3}{4}\right) + 20\left(\frac{7}{8}\right) + 15(0) = \frac{54}{4} + \frac{140}{8} = 13.5 + 17.5 = \mathbf{31}$$

$$\boxed{z^* = w^* = 31 \quad \checkmark \text{ (Dualidad Fuerte verificada)}}$$

**Interpretación económica de los precios sombra:**

- $y_1^* = 3/4 = 0{,}75$: cada litro adicional de **tinta especial** aumenta el margen semanal en **\$0,75 por cien unidades**.
- $y_2^* = 7/8 = 0{,}875$: cada hora adicional de **tiempo de prensa** aumenta el margen en **\$0,875 por cien unidades**.
- $y_3^* = 0$: el **papel recubierto** no es un recurso limitante en el óptimo (la restricción está activa por coincidencia, pero su precio sombra es cero porque el óptimo no cambiaría al agregar más papel).

---

## Ejercicio 2 — Análisis de Sensibilidad

### Contexto de Negocio

**AgroNorte Ltda.** es una empresa agroindustrial que procesa y comercializa dos tipos de conservas: **Conserva de Tomate** ($x_1$, en toneladas) y **Conserva de Pimiento** ($x_2$, en toneladas). El proceso productivo está limitado por la capacidad de dos líneas de procesamiento:

| Recurso | Tomate ($x_1$) | Pimiento ($x_2$) | Disponible |
|:---|:---:|:---:|:---:|
| Línea de esterilización (hrs/ton) | $3$ | $2$ | $\leq 12$ horas |
| Línea de envasado (hrs/ton) | $1$ | $2$ | $\leq 8$ horas |
| **Margen neto (M\$/ton)** | **\$6** | **\$5** | — |

La empresa ha resuelto el siguiente modelo primal:

$$\max \quad z = 6x_1 + 5x_2$$

$$\text{s.a.} \quad \begin{cases}
3x_1 + 2x_2 \leq 12 & \text{(esterilización)} \\
x_1 + 2x_2 \leq 8 & \text{(envasado)} \\
x_1,\, x_2 \geq 0
\end{cases}$$

Resolviendo por el Método Simplex, la solución óptima se obtiene con la base $B = \{x_1, x_2\}$ (ambas variables de decisión son básicas). El **tableau óptimo** ya resuelto es el siguiente:

| Base | $x_1$ | $x_2$ | $s_1$ | $s_2$ | RHS |
|:---:|:---:|:---:|:---:|:---:|:---:|
| **CR** | $0$ | $0$ | $7/4$ | $5/4$ | $\mathbf{26}$ |
| $x_1$ | $1$ | $0$ | $1/2$ | $-1/2$ | $2$ |
| $x_2$ | $0$ | $1$ | $-3/8$ | $3/4$ | $3$ |

La solución óptima es $x_1^* = 2$ ton de Tomate, $x_2^* = 3$ ton de Pimiento, $z^* = \$26$ M (millones).

**Datos del tableau óptimo:**

$$B = \begin{bmatrix} 3 & 2 \\ 1 & 2 \end{bmatrix}, \quad B^{-1} = \begin{bmatrix} 1/2 & -1/2 \\ -1/4 & 3/4 \end{bmatrix}, \quad c_B = [6, 5], \quad b = \begin{bmatrix} 12 \\ 8 \end{bmatrix}$$

**Verificación:** $B^{-1}b = \begin{bmatrix}1/2 & -1/2\\ -1/4 & 3/4\end{bmatrix}\begin{bmatrix}12\\8\end{bmatrix} = \begin{bmatrix}6-4\\-3+6\end{bmatrix} = \begin{bmatrix}2\\3\end{bmatrix}$ ✓

Los **precios sombra** son $y^\top = c_B^\top B^{-1} = [6,5]\begin{bmatrix}1/2 & -1/2\\ -1/4 & 3/4\end{bmatrix} = [3-5/4,\; -3+15/4] = [7/4,\; 5/4]$, consistentes con la fila CR del tableau.

### Parte (a): Análisis de Sensibilidad sobre $b_1$ (horas de esterilización)

Se pregunta: ¿cuánto puede variar la disponibilidad de horas de esterilización ($b_1 = 12$) sin que la base óptima $\{x_1, x_2\}$ deje de ser óptima?

Sea $b_1 \to 12 + \Delta$. La solución básica actualizada es:

$$x_B = B^{-1}(b + \Delta e_1) = B^{-1}b + \Delta B^{-1}e_1 = \begin{bmatrix}2\\3\end{bmatrix} + \Delta \begin{bmatrix}1/2\\-1/4\end{bmatrix}$$

donde $B^{-1}e_1$ es la **primera columna de $B^{-1}$**: $[1/2,\; -1/4]^\top$.

**Condición de factibilidad:** $x_B \geq 0$:

$$x_1 = 2 + \frac{\Delta}{2} \geq 0 \quad \Rightarrow \quad \Delta \geq -4$$

$$x_2 = 3 - \frac{\Delta}{4} \geq 0 \quad \Rightarrow \quad \Delta \leq 12$$

**Rango de optimalidad para $b_1$:**

$$-4 \leq \Delta \leq 12 \quad \Rightarrow \quad 12 - 4 \leq b_1 \leq 12 + 12 \quad \Rightarrow \quad \boxed{8 \leq b_1 \leq 24}$$

**Interpretación:** La empresa puede aumentar las horas de esterilización hasta **24 horas** o reducirlas hasta **8 horas** sin que cambie la composición de la solución óptima (ambas conservas seguirán siendo producidas). El precio sombra $y_1^* = 7/4 = 1{,}75$ M\$/hora es válido en todo este rango.

### Parte (b): Análisis de Sensibilidad sobre $b_2$ (horas de envasado)

Sea $b_2 \to 8 + \Delta$. La solución básica actualizada es:

$$x_B = B^{-1}b + \Delta B^{-1}e_2 = \begin{bmatrix}2\\3\end{bmatrix} + \Delta \begin{bmatrix}-1/2\\3/4\end{bmatrix}$$

donde $B^{-1}e_2$ es la **segunda columna de $B^{-1}$**: $[-1/2,\; 3/4]^\top$.

**Condición de factibilidad:** $x_B \geq 0$:

$$x_1 = 2 - \frac{\Delta}{2} \geq 0 \quad \Rightarrow \quad \Delta \leq 4$$

$$x_2 = 3 + \frac{3\Delta}{4} \geq 0 \quad \Rightarrow \quad \Delta \geq -4$$

**Rango de optimalidad para $b_2$:**

$$-4 \leq \Delta \leq 4 \quad \Rightarrow \quad 8 - 4 \leq b_2 \leq 8 + 4 \quad \Rightarrow \quad \boxed{4 \leq b_2 \leq 12}$$

**Interpretación:** La capacidad de la línea de envasado puede fluctuar entre **4 y 12 horas** sin que cambie la base óptima. El precio sombra $y_2^* = 5/4 = 1{,}25$ M\$/hora se mantiene válido en este rango.

### Parte (c): Análisis de Sensibilidad sobre $c_1$ (margen de Conserva de Tomate)

Se pregunta: ¿en qué rango puede variar el margen neto de la Conserva de Tomate ($c_1 = 6$) sin que cambie la base óptima?

$x_1$ es una variable **básica** (está en la base), por lo que cambiar $c_1$ afecta a $c_B$ y modifica todos los costos reducidos de las variables no básicas.

Sea $c_1 \to 6 + \Delta$. El nuevo vector $c_B = [6 + \Delta, 5]$.

Los costos reducidos de las no básicas $s_1$ y $s_2$ (columnas de $B^{-1}A_{NB}$ son las columnas de $B^{-1}$ para las holguras, que son exactamente las columnas de $B^{-1}$ misma):

$$\bar{c}_{s_1} = c_B^\top B^{-1} e_{s_1} - 0 = [6+\Delta, 5] \begin{bmatrix}1/2\\-1/4\end{bmatrix} = \frac{6+\Delta}{2} - \frac{5}{4} = 3 + \frac{\Delta}{2} - \frac{5}{4} = \frac{7}{4} + \frac{\Delta}{2}$$

$$\bar{c}_{s_2} = c_B^\top B^{-1} e_{s_2} - 0 = [6+\Delta, 5] \begin{bmatrix}-1/2\\3/4\end{bmatrix} = -\frac{6+\Delta}{2} + \frac{15}{4} = -3 - \frac{\Delta}{2} + \frac{15}{4} = \frac{3}{4} - \frac{\Delta}{2}$$

**Condición de optimalidad:** $\bar{c}_{s_1} \geq 0$ y $\bar{c}_{s_2} \geq 0$:

$$\frac{7}{4} + \frac{\Delta}{2} \geq 0 \quad \Rightarrow \quad \Delta \geq -\frac{7}{2} = -3{,}5$$

$$\frac{3}{4} - \frac{\Delta}{2} \geq 0 \quad \Rightarrow \quad \Delta \leq \frac{3}{2} = 1{,}5$$

**Rango de optimalidad para $c_1$:**

$$-3{,}5 \leq \Delta \leq 1{,}5 \quad \Rightarrow \quad \boxed{2{,}5 \leq c_1 \leq 7{,}5}$$

**Interpretación:** Si el margen neto por tonelada de Conserva de Tomate se mantiene entre **\$2,5 M y \$7,5 M por tonelada**, la combinación óptima ($x_1^* = 2$ ton de Tomate y $x_2^* = 3$ ton de Pimiento) seguirá siendo la solución óptima. Fuera de este rango, convendrá cambiar la mezcla de producción.

### Parte (d): Interpretación de los Precios Sombra en el Contexto de AgroNorte

Los precios sombra obtenidos del tableau óptimo son:

$$y_1^* = \frac{7}{4} = 1{,}75 \text{ M\$/hora}, \qquad y_2^* = \frac{5}{4} = 1{,}25 \text{ M\$/hora}$$

**Significado operacional:**

- **$y_1^* = 1{,}75$ M\$/hora:** Si AgroNorte pudiese contratar **una hora extra** de capacidad en la línea de esterilización (por ejemplo, mediante horas extras o subcontratación), el margen semanal total aumentaría en **\$1,75 millones**. Este es el precio máximo que la empresa debería estar dispuesta a pagar por esa hora adicional.

- **$y_2^* = 1{,}25$ M\$/hora:** Análogamente, una hora adicional en la línea de envasado generaría **\$1,25 millones** adicionales de margen. Por tanto, la línea de esterilización es el cuello de botella más valioso.

- **Rango de validez:** estos precios sombra son válidos únicamente dentro de los rangos calculados: $b_1 \in [8, 24]$ horas y $b_2 \in [4, 12]$ horas. Si la empresa proyecta ampliar la capacidad de esterilización más allá de 24 horas, el precio sombra cambiará y se deberá resolver el problema nuevamente.

- **Decisión de inversión:** si la empresa puede arrendar una hora adicional de esterilización a un costo inferior a \$1,75 M, la inversión es rentable. Si el costo supera \$1,75 M, no conviene.

---

## Ejercicio 3 — Dual Completo y Sensibilidad Combinada

### Contexto de Negocio

**LogiCarga S.A.** es una empresa de logística urbana que reparte pedidos usando dos tipos de vehículos: **Furgones** ($x_1$, en decenas de viajes diarios) y **Motos** ($x_2$, en decenas de viajes diarios). La empresa quiere **minimizar el costo operacional diario** sujeto a compromisos mínimos de servicio:

| Condición | Furgones ($x_1$) | Motos ($x_2$) | Requerimiento |
|:---|:---:|:---:|:---:|
| Pedidos grandes cubiertos (uds/decena viajes) | $3$ | $1$ | $\geq 9$ unidades |
| Pedidos urgentes cubiertos (uds/decena viajes) | $1$ | $2$ | $\geq 8$ unidades |
| **Costo (M\$/decena de viajes)** | **\$4** | **\$3** | — |

### Formulación Primal

$$\min \quad z = 4x_1 + 3x_2$$

$$\text{s.a.} \quad \begin{cases}
3x_1 + x_2 \geq 9 & \text{(pedidos grandes)} \\
x_1 + 2x_2 \geq 8 & \text{(pedidos urgentes)} \\
x_1,\, x_2 \geq 0
\end{cases}$$

### Parte (a): Plantee el Dual

El primal es de **minimización** con restricciones $\geq$ y variables $\geq 0$. Usando la tabla de dualidad:

- Cada restricción $\geq b_i$ genera una variable dual $y_i \geq 0$.
- Cada variable $x_j \geq 0$ genera una restricción dual $\leq c_j$.
- El dual de un problema de minimización es de **maximización**.

$$\max \quad w = 9y_1 + 8y_2$$

$$\text{s.a.} \quad \begin{cases}
3y_1 + y_2 \leq 4 & \text{(columna de } x_1\text{)} \\
y_1 + 2y_2 \leq 3 & \text{(columna de } x_2\text{)} \\
y_1,\, y_2 \geq 0
\end{cases}$$

> **Interpretación:** $y_1$ y $y_2$ representan el valor (precio sombra) de cada unidad de pedido grande y pedido urgente, respectivamente. El dual busca **maximizar el valor imputado** a los compromisos de servicio.

### Parte (b): Resuelva el Primal por Simplex Dual (Tableau)

El primal de minimización con restricciones $\geq$ admite resolución directa por **Simplex Dual**: las holguras (negativas) dan una solución dual-factible.

**Forma estándar** (se restan excedentes $s_1, s_2 \geq 0$ y se multiplica por $-1$ para llevar al formato de Simplex Dual):

$$\begin{cases}
-3x_1 - x_2 + s_1 = -9 \\
-x_1 - 2x_2 + s_2 = -8
\end{cases}$$

**Tableau Inicial (Iteración 0)**

Base inicial: $\{s_1, s_2\}$. Para minimización, la fila $z$ muestra los coeficientes de la función objetivo. Todos son positivos ($4, 3 \geq 0$): dual-factible ✓.

| Base | $x_1$ | $x_2$ | $s_1$ | $s_2$ | RHS |
|:---:|:---:|:---:|:---:|:---:|:---:|
| **$z$** | $4$ | $3$ | $0$ | $0$ | $0$ |
| $s_1$ | $-3$ | $-1$ | $1$ | $0$ | $-9$ |
| $s_2$ | $-1$ | $-2$ | $0$ | $1$ | $-8$ |

**Criterio de salida:** el RHS más negativo es $-9$ (fila $s_1$) $\Rightarrow$ **sale $s_1$**.

**Criterio de entrada** (columnas con $a_{1j} < 0$):

$$\min\left(\left|\frac{4}{-3}\right|,\; \left|\frac{3}{-1}\right|\right) = \min\left(\frac{4}{3},\; 3\right) = \frac{4}{3}$$

**Entra $x_1$**. Elemento pivote: $\boxed{-3}$.

#### Iteración 1

**Operaciones de fila** (pivote $= -3$, fila de $s_1$):

$$R_{x_1}^{\text{nuevo}} = \frac{R_{s_1}}{-3} = \left[1,\; \frac{1}{3},\; -\frac{1}{3},\; 0 \;\middle|\; 3\right]$$

$$R_z^{\text{nuevo}} = R_z - 4 \cdot R_{x_1}^{\text{nuevo}} = [4,3,0,0,0] - 4\cdot[1,\tfrac{1}{3},-\tfrac{1}{3},0,3]$$
$$= [4-4,\;3-\tfrac{4}{3},\;0+\tfrac{4}{3},\;0,\;0-12] = [0,\;\tfrac{5}{3},\;\tfrac{4}{3},\;0,\;-12]$$

$$R_{s_2}^{\text{nuevo}} = R_{s_2} - (-1) \cdot R_{x_1}^{\text{nuevo}} = [-1,-2,0,1,-8] + [1,\tfrac{1}{3},-\tfrac{1}{3},0,3]$$
$$= [0,\;-\tfrac{5}{3},\;-\tfrac{1}{3},\;1,\;-5]$$

| Base | $x_1$ | $x_2$ | $s_1$ | $s_2$ | RHS |
|:---:|:---:|:---:|:---:|:---:|:---:|
| **$z$** | $0$ | $5/3$ | $4/3$ | $0$ | $\mathbf{-12}$ |
| $x_1$ | $1$ | $1/3$ | $-1/3$ | $0$ | $3$ |
| $s_2$ | $0$ | $-5/3$ | $-1/3$ | $1$ | $-5$ |

RHS de $s_2 = -5 < 0$ $\Rightarrow$ **sale $s_2$**.

**Criterio de entrada** (columnas con $a_{2j} < 0$ en fila de $s_2$):

$$\min\left(\left|\frac{5/3}{-5/3}\right|,\; \left|\frac{4/3}{-1/3}\right|\right) = \min(1,\; 4) = 1$$

**Entra $x_2$**. Elemento pivote: $\boxed{-5/3}$.

#### Iteración 2 — Solución Óptima

**Operaciones de fila** (pivote $= -5/3$, fila de $s_2$):

$$R_{x_2}^{\text{nuevo}} = \frac{R_{s_2}}{-5/3} = \frac{3}{-5} \cdot R_{s_2} = \left[0,\; 1,\; \frac{1}{5},\; -\frac{3}{5} \;\middle|\; 3\right]$$

$$R_z^{\text{nuevo}} = R_z - \frac{5}{3} \cdot R_{x_2}^{\text{nuevo}} = [0,\tfrac{5}{3},\tfrac{4}{3},0,-12] - \tfrac{5}{3}\cdot[0,1,\tfrac{1}{5},-\tfrac{3}{5},3]$$
$$= [0,\;\tfrac{5}{3}-\tfrac{5}{3},\;\tfrac{4}{3}-\tfrac{1}{3},\;0+1,\;-12-5] = [0,\;0,\;1,\;1,\;\mathbf{-17}]$$

$$R_{x_1}^{\text{nuevo}} = R_{x_1} - \frac{1}{3} \cdot R_{x_2}^{\text{nuevo}} = [1,\tfrac{1}{3},-\tfrac{1}{3},0,3] - \tfrac{1}{3}\cdot[0,1,\tfrac{1}{5},-\tfrac{3}{5},3]$$
$$= [1,\;\tfrac{1}{3}-\tfrac{1}{3},\;-\tfrac{1}{3}-\tfrac{1}{15},\;0+\tfrac{1}{5},\;3-1] = [1,\;0,\;-\tfrac{2}{5},\;\tfrac{1}{5},\;2]$$

| Base | $x_1$ | $x_2$ | $s_1$ | $s_2$ | RHS |
|:---:|:---:|:---:|:---:|:---:|:---:|
| **$z$** | $0$ | $0$ | $1$ | $1$ | $\mathbf{-17}$ |
| $x_1$ | $1$ | $0$ | $-2/5$ | $1/5$ | $2$ |
| $x_2$ | $0$ | $1$ | $1/5$ | $-3/5$ | $3$ |

**Todos los RHS son** $\geq 0$ ($x_1 = 2 > 0$, $x_2 = 3 > 0$): solución factible ✓.  
**Todos los coeficientes en la fila $z$ son** $\geq 0$ ($1, 1 \geq 0$): dual-factible ✓.

$$\boxed{x_1^* = 2 \text{ decenas de viajes Furgón}, \quad x_2^* = 3 \text{ decenas de viajes Moto}, \quad z^* = 17 \text{ M\$}}$$

> **Nota sobre el signo de $z$:** el valor del RHS en la fila $z$ es $-17$ por la convención del tableau. El costo mínimo real es $z^* = 17$. Verificación: $z^* = 4(2) + 3(3) = 8 + 9 = 17$ ✓.

### Parte (c): Análisis de Sensibilidad sobre $b_1$ (pedidos grandes)

Del tableau óptimo, la base es $B = \{x_1, x_2\}$ con columnas originales:

$$B = \begin{bmatrix} 3 & 1 \\ 1 & 2 \end{bmatrix}$$

La inversa (leída directamente de las columnas de $s_1$ y $s_2$ en el tableau óptimo, **con signos invertidos** por la convención de holguras negativas):

$$B^{-1} = \begin{bmatrix} 2/5 & -1/5 \\ -1/5 & 3/5 \end{bmatrix}$$

**Verificación:** $B B^{-1} = \begin{bmatrix}3&1\\1&2\end{bmatrix}\begin{bmatrix}2/5&-1/5\\-1/5&3/5\end{bmatrix} = \begin{bmatrix}6/5-1/5 & -3/5+3/5\\2/5-2/5 & -1/5+6/5\end{bmatrix} = \begin{bmatrix}1&0\\0&1\end{bmatrix}$ ✓

Sea $b_1 \to 9 + \Delta$:

$$x_B = B^{-1}b + \Delta B^{-1}e_1 = \begin{bmatrix}2\\3\end{bmatrix} + \Delta \begin{bmatrix}2/5\\-1/5\end{bmatrix}$$

**Condición de factibilidad:**

$$x_1 = 2 + \frac{2\Delta}{5} \geq 0 \quad \Rightarrow \quad \Delta \geq -5$$

$$x_2 = 3 - \frac{\Delta}{5} \geq 0 \quad \Rightarrow \quad \Delta \leq 15$$

**Rango de optimalidad para $b_1$:**

$$\boxed{4 \leq b_1 \leq 24}$$

**Precio sombra:** $y_1^* = c_B^\top (B^{-1})_1 = [4,3][2/5, -1/5]^\top = 8/5 - 3/5 = 1$ M\$/unidad de pedido grande. Este es el ahorro en costo por cada unidad adicional de pedido grande que la empresa logística pueda cubrir con la flota actual.

### Parte (d): Verificación de Dualidad Fuerte y Precios Sombra

La solución del dual (problema de maximización con restricciones $\leq$):

Los precios sombra del primal son exactamente los valores duales óptimos. Del tableau óptimo del primal, la fila $z$ muestra los costos reducidos de $s_1$ y $s_2$, que corresponden a los valores duales:

$$y_1^* = 1 \quad (\text{costo reducido de } s_1), \qquad y_2^* = 1 \quad (\text{costo reducido de } s_2)$$

**Verificación de factibilidad dual:**

$$3y_1^* + y_2^* = 3(1) + 1 = 4 \leq 4 \; ✓$$
$$y_1^* + 2y_2^* = 1 + 2 = 3 \leq 3 \; ✓$$

**Dualidad fuerte:**

$$z^* = 4(2) + 3(3) = 17$$
$$w^* = 9(1) + 8(1) = 17$$

$$\boxed{z^* = w^* = 17 \text{ M\$} \quad \checkmark}$$

**Holguras complementarias verificadas:**

- Restricción 1 primal: $3(2) + 1(3) = 9 = 9$ (activa) $\Rightarrow$ $y_1^* = 1 \neq 0$ ✓
- Restricción 2 primal: $1(2) + 2(3) = 8 = 8$ (activa) $\Rightarrow$ $y_2^* = 1 \neq 0$ ✓
- $x_1^* = 2 > 0$ $\Rightarrow$ restricción dual 1 activa: $3(1) + 1 = 4 = 4$ ✓
- $x_2^* = 3 > 0$ $\Rightarrow$ restricción dual 2 activa: $1 + 2(1) = 3 = 3$ ✓

**Todas las condiciones de holgura complementaria se satisfacen.** La empresa LogiCarga debe operar **20 viajes de Furgón y 30 viajes de Moto** (en decenas) al día con un costo mínimo de **\$17 millones**.

---

## Resumen de la Ayudantía

| Concepto | Idea Clave | Herramienta |
|:---|:---|:---|
| **Problema Dual** | A todo primal le corresponde un dual; variables duales son precios sombra | Tabla de dualidad |
| **Dualidad Débil** | $b^\top y \leq c^\top x$ para cualquier par factible | Cota |
| **Dualidad Fuerte** | $z^* = w^*$ en el óptimo | Verificación |
| **Simplex Dual** | Sale el RHS más negativo; entra el cociente $|w_j / a_{rj}|$ mínimo | Tableau |
| **Holguras Complementarias** | $y_i^* (a_i^\top x^* - b_i) = 0$ y $x_j^* (c_j - a_j^\top y^*) = 0$ | Recuperar primal desde dual |
| **Sensibilidad en $b$** | Rango de $\Delta$ tal que $B^{-1}(b + \Delta e_i) \geq 0$ | Análisis post-óptimo |
| **Sensibilidad en $c$** | Rango de $\Delta$ tal que todos $\bar{c}_j \geq 0$ con nuevo $c_k$ | Análisis post-óptimo |
| **Precio Sombra** | $y_i^* = \partial z^* / \partial b_i$ — válido solo en el rango de sensibilidad | Decisiones de inversión |

---

*Ayudantía 8 — IIP314W Optimización Aplicada a Negocios — Universidad del Desarrollo — 2026-T1*